In [ ]:
import altair as alt
import marimo as mo
import polars as pl

# Loading lives in plain modules, so it can be imported, tested, and
# type-checked without marimo's cell semantics in the way.
from reader import load_run, load_truth, with_format
from session import find_runs, open_session, summary_table

# Benchmark run

A corpus of synthetic documents with known planted values, what a
pipeline reported finding in them, and — where a report has been
written — what became of each one.

Scoring happens in TypeScript, next to the types that define what a match
is. Run `synthetic score --report ./report` and the scored sections
below fill in; without one, the raw sides are still shown side by side.

In [ ]:
runs = find_runs()
run_picker = mo.ui.dropdown(
    options={path.name: path for path in runs},
    value=runs[0].name if runs else None,
    label="Run",
)
run_picker if runs else mo.md(
    "**No runs found.** Generate a corpus and run `run` first."
)

&lt;marimo-dropdown data-initial-value=&#x27;[&amp;quot;5-mtrskkbz&amp;quot;]&#x27; data-label=&#x27;&amp;quot;&amp;#92;u003cspan class=&amp;#92;&amp;quot;markdown prose dark:prose-invert contents&amp;#92;&amp;quot;&amp;#92;u003e&amp;#92;u003cspan class=&amp;#92;&amp;quot;paragraph&amp;#92;&amp;quot;&amp;#92;u003eRun&amp;#92;u003c/span&amp;#92;u003e&amp;#92;u003c/span&amp;#92;u003e&amp;quot;&#x27; data-options=&#x27;[&amp;quot;5-mtrskkbz&amp;quot;]&#x27; data-allow-select-none=&#x27;false&#x27; data-searchable=&#x27;false&#x27; data-full-width=&#x27;false&#x27; data-disabled=&#x27;false&#x27;&gt;&lt;/marimo-dropdown&gt;

In [ ]:
# Stops here when there is no run, leaving the message above readable
# rather than replacing it with the exception `open_session` would raise.
mo.stop(
    not run_picker.options,
    mo.md("*Nothing to explore until a run exists.*"),
)

session = open_session(run_picker.value)
mo.md(summary_table(session))

| | |
|---|---|
| Run | `5-mtrskkbz` |
| Corpus seed | 5 |
| Pipeline | `benchmark` |
| Records | 12 |
| Labels in scope | 21 |
| Started | 2026-09-07T22:07:53.327Z |
| Scored | no — run `synthetic score` |

In [ ]:
planted = load_truth(session.corpus_dir)
found = with_format(load_run(session.run_dir), planted)

## Raw counts

What was planted against what came back, per label, with nothing matched
up. A label with equal counts on both sides has not necessarily been
scored well — the detections may sit on the wrong values entirely. Useful
when there is no report, and as a check that the scorer saw what the run
actually holds.

In [ ]:
planted_by_label = planted.group_by("label").agg(
    pl.len().alias("planted"),
    (pl.col("expect") == "ignored").sum().alias("must_not_detect"),
)
found_by_label = (
    found.filter(pl.col("label").is_not_null())
    .group_by("label")
    .agg(pl.len().alias("detected"))
)

by_label = (
    planted_by_label.join(found_by_label, on="label", how="full", coalesce=True)
    .fill_null(0)
    .sort("planted", descending=True)
)
by_label

label,planted,must_not_detect,detected
str,u32,u32,u32
"""person_name""",33,0,0
"""bank_account""",18,11,6
"""email_address""",18,0,19
"""phone_number""",11,1,2
"""payment_card""",10,1,8
…,…,…,…
"""username""",2,0,0
"""insurance_id""",1,0,0
"""street_address""",1,0,0


In [ ]:
chart_data = by_label.unpivot(
    index="label",
    on=["planted", "detected"],
    variable_name="side",
    value_name="count",
).filter(pl.col("count") > 0)

mo.ui.altair_chart(
    alt.Chart(chart_data)
    .mark_bar()
    .encode(
        y=alt.Y("label:N", sort="-x", title=None),
        x=alt.X("count:Q", title="occurrences"),
        yOffset="side:N",
        color=alt.Color("side:N", title=None),
        tooltip=["label", "side", "count"],
    )
    .properties(height=alt.Step(12))
)

## By format

Where a pipeline reads a document differently, this is where it shows.
A format whose planted values never come back is usually one whose text
the pipeline did not extract, rather than one whose values it failed to
recognise.

In [ ]:
by_format = (
    planted.group_by("format")
    .agg(pl.len().alias("planted"), pl.col("record").n_unique().alias("records"))
    .join(
        found.filter(pl.col("label").is_not_null())
        .group_by("format")
        .agg(pl.len().alias("detected")),
        on="format",
        how="left",
    )
    .fill_null(0)
    .with_columns(
        (pl.col("detected") / pl.col("planted")).round(2).alias("ratio"),
    )
    .sort("planted", descending=True)
)
by_format

format,planted,records,detected,ratio
str,u32,u32,u32,f64
"""txt""",83,7,30,0.36
"""csv""",29,2,14,0.48
"""json""",24,2,11,0.46
"""xml""",13,1,7,0.54


## One record

The planted values and the detections for a single document, so a
disagreement can be looked at directly rather than inferred from counts.

In [ ]:
record_ids = planted.select(pl.col("record").unique().sort()).to_series().to_list()
record_picker = mo.ui.dropdown(
    options=record_ids,
    value=record_ids[0] if record_ids else None,
    label="Record",
)
record_picker

&lt;marimo-dropdown data-initial-value=&#x27;[&amp;quot;rec_0001&amp;quot;]&#x27; data-label=&#x27;&amp;quot;&amp;#92;u003cspan class=&amp;#92;&amp;quot;markdown prose dark:prose-invert contents&amp;#92;&amp;quot;&amp;#92;u003e&amp;#92;u003cspan class=&amp;#92;&amp;quot;paragraph&amp;#92;&amp;quot;&amp;#92;u003eRecord&amp;#92;u003c/span&amp;#92;u003e&amp;#92;u003c/span&amp;#92;u003e&amp;quot;&#x27; data-options=&#x27;[&amp;quot;rec_0001&amp;quot;,&amp;quot;rec_0002&amp;quot;,&amp;quot;rec_0003&amp;quot;,&amp;quot;rec_0004&amp;quot;,&amp;quot;rec_0005&amp;quot;,&amp;quot;rec_0006&amp;quot;,&amp;quot;rec_0007&amp;quot;,&amp;quot;rec_0008&amp;quot;,&amp;quot;rec_0009&amp;quot;,&amp;quot;rec_0010&amp;quot;,&amp;quot;rec_0011&amp;quot;,&amp;quot;rec_0012&amp;quot;]&#x27; data-allow-select-none=&#x27;false&#x27; data-searchable=&#x27;false&#x27; data-full-width=&#x27;false&#x27; data-disabled=&#x27;false&#x27;&gt;&lt;/marimo-dropdown&gt;

In [ ]:
# Stacked, because only a cell's last expression renders: a bare `mo.md`
# above the table is discarded, leaving two adjacent tables whose meaning
# has to be inferred from their columns.
mo.vstack(
    [
        mo.md("**Planted**"),
        planted.filter(pl.col("record") == record_picker.value)
        .select("label", "surface", "expect", "adversarial", "start", "end", "text")
        .sort("start"),
    ]
)

label,surface,expect,adversarial,start,end,text
str,str,str,str,i64,i64,str
"""organization_name""","""canonical""",null,null,25,34,"""Price Inc"""
"""person_name""","""canonical""",null,"""common_word""",76,86,"""April Case"""
"""case_number""","""canonical""",null,"""format_collision""",100,111,"""471-88-2130"""
"""person_name""","""abbreviated""",null,"""common_word""",158,165,"""A. Case"""
"""ip_address""","""canonical""",null,"""unusual_context""",274,286,"""102.22.21.39"""
…,…,…,…,…,…,…
"""email_address""","""canonical""",null,null,1356,1378,"""Ayden60+OAfm@yahoo.com"""
"""person_name""","""misspelled""",null,"""common_word""",1459,1469,"""April aCse"""
"""case_number""","""canonical""","""ignored""","""format_collision""",1564,1578,"""CVE-2026-31337"""


In [ ]:
mo.vstack(
    [
        mo.md("**Detected**"),
        found.filter(pl.col("record") == record_picker.value)
        .select("label", "confidence", "recognizer", "start", "end", "failed")
        .sort("start"),
    ]
)

label,confidence,recognizer,start,end,failed
str,f64,str,i64,i64,str
"""government_id""",0.5,"""pattern""",100,111,null
"""ip_address""",0.6,"""pattern""",274,286,null
"""ip_address""",0.6,"""pattern""",633,650,null
"""email_address""",0.5,"""pattern""",1356,1378,null


## Values that must survive

A corpus plants decoys — a catalog number shaped like an account, a
published switchboard, a public CVE — that a pipeline is expected to
leave alone. A detection overlapping one of these is over-redaction, and
the reason a benchmark that measures only recall is not a benchmark.

In [ ]:
planted.filter(pl.col("expect") == "ignored").select(
    "record", "label", "adversarial", "text"
)

record,label,adversarial,text
str,str,str,str
"""rec_0001""","""case_number""","""format_collision""","""CVE-2026-31337"""
"""rec_0001""","""device_id""","""format_collision""","""8.4.1-rc2+build.20260311"""
"""rec_0002""","""payment_card""","""format_collision""","""4111 1111 1111 1112"""
"""rec_0002""","""iban""","""format_collision""","""GB29 NWBK 6016 1331 9268 000"""
"""rec_0003""","""bank_account""","""unusual_context""","""000987654321"""
…,…,…,…
"""rec_0011""","""bank_account""","""format_collision""","""000987654321"""
"""rec_0011""","""bank_account""","""format_collision""","""000555000555"""
"""rec_0011""","""bank_account""","""format_collision""","""000456000456"""
